In [2]:
import pandas as pd
from sqlalchemy import create_engine
import pymysql

In [3]:
# Please refer to BED file format
# https://en.wikipedia.org/wiki/BED_(file_format)

def load_data(file_name, type):
    columns = ['chrom', 'chromStart', 'chromEnd', 'name']
    df = pd.read_csv(file_name, sep="\t", header=None, names=columns)
    df.loc[:, "length"] = df["chromEnd"] - df["chromStart"]
    df.loc[:, "type"] = type
    return df

## Load the Histone Dataset

In [4]:
h3k4me3_df = load_data("dataset/HepG2_Male.histone.H3K4me3.peak.bed", "h3k4me3")
h3k9ac_df = load_data("dataset/HepG2_Male.histone.H3K9ac.peak.bed", "h3k9ac")
h3k27ac_df = load_data("dataset/HepG2_Male.histone.H3K27ac.peak.bed", "h3k27ac")
h3k27me3_df = load_data("dataset/HepG2_Male.histone.H3K27me3.peak.bed", "h3k27me3")
h3k9me3_df = load_data("dataset/HepG2_Male.histone.H3K9me3.peak.bed", "h3k9me3")

In [5]:
frames = [h3k4me3_df, h3k9ac_df, h3k27ac_df, h3k27me3_df, h3k9me3_df]
histone_df = pd.concat(frames)
histone_df.head()

,chrom,chromStart,chromEnd,name,length,type
0,chr10,119808,119954,chr10_173,146,h3k4me3
1,chr10,119956,120102,chr10_174,146,h3k4me3
2,chr10,122100,122246,chr10_185,146,h3k4me3
3,chr10,122308,122454,chr10_186,146,h3k4me3
4,chr10,180346,180492,chr10_489,146,h3k4me3


In [47]:
print(histone_df.shape)
histone_df.describe()

(319086, 6)


,chromStart,chromEnd,length
count,3.190860e+05,3.190860e+05,319086.0
mean,7.568847e+07,7.568862e+07,146.0
std,5.573188e+07,5.573188e+07,0.0
min,6.893000e+03,7.039000e+03,146.0
25%,3.228027e+07,3.228041e+07,146.0
50%,6.304105e+07,6.304120e+07,146.0
75%,1.112144e+08,1.112145e+08,146.0
max,2.492390e+08,2.492392e+08,146.0


## Load refseq dataset from database

In [7]:
username = "root"
password = ""
port = 3306
database = "hg38"

In [8]:
engine = create_engine('mysql+pymysql://%s@localhost:%i/%s' %(username, port, database))

In [9]:
sql = "SELECT * FROM ncbirefseq"
df = pd.read_sql_query(sql, engine)

df.head()

,bin,name,chrom,strand,txStart,txEnd,cdsStart,cdsEnd,exonCount,exonStarts,exonEnds,score,name2,cdsStartStat,cdsEndStat,exonFrames
0,585,NR_046018.2,chr1,+,11873,14409,14409,14409,3,"b'11873,12612,13220,'","b'12227,12721,14409,'",0,DDX11L1,none,none,"b'-1,-1,-1,'"
1,585,NR_024540.1,chr1,-,14361,29370,29370,29370,11,"b'14361,14969,15795,16606,16857,17232,17605,17...","b'14829,15038,15947,16765,17055,17368,17742,18...",0,WASH7P,none,none,"b'-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,'"
2,585,NR_106918.1,chr1,-,17368,17436,17436,17436,1,"b'17368,'","b'17436,'",0,MIR6859-1,none,none,"b'-1,'"
3,585,XR_007065314.1,chr1,+,29773,35418,35418,35418,3,"b'29773,30975,34167,'","b'30667,31093,35418,'",0,MIR1302-2HG,none,none,"b'-1,-1,-1,'"
4,585,NR_036051.1,chr1,+,30365,30503,30503,30503,1,"b'30365,'","b'30503,'",0,MIR1302-2,none,none,"b'-1,'"


In [10]:
df.shape

(196097, 16)

### Select the necessary columns

In [11]:
refseq_df = df[["bin", "name", "chrom", "strand", "txStart", "txEnd"]]
refseq_df.head()

,bin,name,chrom,strand,txStart,txEnd
0,585,NR_046018.2,chr1,+,11873,14409
1,585,NR_024540.1,chr1,-,14361,29370
2,585,NR_106918.1,chr1,-,17368,17436
3,585,XR_007065314.1,chr1,+,29773,35418
4,585,NR_036051.1,chr1,+,30365,30503


### Find the TSS

In [12]:
def get_tss(row):
    if row['strand'] == "+":
        return row['txStart']
    return row['txEnd']

In [16]:
refseq_df.loc[:, "tss"] = refseq_df.apply(lambda row: get_tss(row), axis=1)
refseq_df.head()

,bin,name,chrom,strand,txStart,txEnd,tss
0,585,NR_046018.2,chr1,+,11873,14409,11873
1,585,NR_024540.1,chr1,-,14361,29370,29370
2,585,NR_106918.1,chr1,-,17368,17436,17436
3,585,XR_007065314.1,chr1,+,29773,35418,29773
4,585,NR_036051.1,chr1,+,30365,30503,30365


## Histone counting inside +/-2k from TSS

In [33]:
# Example for gene ID: NR_036051.1
tss_test = 30365
threshold = 0.8
histone_length = 146

hist_df = histone_df[((histone_df["chromStart"] >= tss_test - 2000) & 
                     (histone_df["chromEnd"] <= tss_test + 2000)) |
                     (((histone_df["chromEnd"] - (tss_test - 2000))/histone_length).between(threshold, 1.0)) |
                     ((((tss_test + 2000) - histone_df["chromStart"])/histone_length).between(threshold, 1.0))]

print(hist_df.shape)
print(len(hist_df.index))
hist_df.head()

(9, 6)
9


,chrom,chromStart,chromEnd,name,length,type
23544,chr17,30588,30734,chr17_149,146,h3k4me3
76691,chr9,28898,29044,chr9_13,146,h3k4me3
21369,chr17,30193,30339,chr17_148,146,h3k9ac
68485,chr9,28898,29044,chr9_13,146,h3k9ac
85644,chr9,28898,29044,chr9_13,146,h3k27ac


In [39]:
def count_histone(row, threshold = 0.8, histone_length = 146):
    tss = row["tss"]
    hist_df = histone_df[((histone_df["chromStart"] >= tss - 2000) & 
                     (histone_df["chromEnd"] <= tss + 2000)) |
                     (((histone_df["chromEnd"] - (tss - 2000))/histone_length).between(threshold, 1.0)) |
                     ((((tss + 2000) - histone_df["chromStart"])/histone_length).between(threshold, 1.0))]
    
    return len(hist_df.index)

In [40]:
# slice the first 10,000 rows
refseq10000_df = refseq_df[:10000]
print(refseq10000_df.shape)
refseq10000_df.head()

(10000, 7)


,bin,name,chrom,strand,txStart,txEnd,tss
0,585,NR_046018.2,chr1,+,11873,14409,11873
1,585,NR_024540.1,chr1,-,14361,29370,29370
2,585,NR_106918.1,chr1,-,17368,17436,17436
3,585,XR_007065314.1,chr1,+,29773,35418,29773
4,585,NR_036051.1,chr1,+,30365,30503,30365


In [42]:
refseq10000_df.loc[:, "count_histone"] = refseq10000_df.apply(lambda row: count_histone(row), axis=1)
refseq10000_df.head()

C:\Users\abdul\AppData\Local\Temp\ipykernel_8584\4087983205.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  refseq10000_df.loc[:, "count_histone"] = refseq10000_df.apply(lambda row: count_histone(row), axis=1)


,bin,name,chrom,strand,txStart,txEnd,tss,count_histone
0,585,NR_046018.2,chr1,+,11873,14409,11873,6
1,585,NR_024540.1,chr1,-,14361,29370,29370,9
2,585,NR_106918.1,chr1,-,17368,17436,17436,0
3,585,XR_007065314.1,chr1,+,29773,35418,29773,9
4,585,NR_036051.1,chr1,+,30365,30503,30365,9


In [43]:
refseq10000_df.to_csv("dataset/results/refseq10000.csv", index=False)

In [44]:
refseq_df.loc[:, "count_histone"] = refseq_df.apply(lambda row: count_histone(row), axis=1)
refseq_df.head()

refseq_df.to_csv("dataset/results/refseq_histone_count.csv", index=False)

C:\Users\abdul\AppData\Local\Temp\ipykernel_8584\3126422563.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  refseq_df.loc[:, "count_histone"] = refseq_df.apply(lambda row: count_histone(row), axis=1)


In [48]:
refseq_df.head()

,bin,name,chrom,strand,txStart,txEnd,tss,count_histone
0,585,NR_046018.2,chr1,+,11873,14409,11873,6
1,585,NR_024540.1,chr1,-,14361,29370,29370,9
2,585,NR_106918.1,chr1,-,17368,17436,17436,0
3,585,XR_007065314.1,chr1,+,29773,35418,29773,9
4,585,NR_036051.1,chr1,+,30365,30503,30365,9


In [49]:
refseq_df.describe()

,bin,txStart,txEnd,tss,count_histone
count,196097.000000,1.960970e+05,1.960970e+05,1.960970e+05,196097.000000
mean,712.968011,6.827849e+07,6.835499e+07,6.831565e+07,8.326124
std,577.773176,5.751824e+07,5.752883e+07,5.752260e+07,9.738842
min,0.000000,0.000000e+00,2.950000e+02,0.000000e+00,0.000000
25%,146.000000,2.162809e+07,2.166936e+07,2.166936e+07,1.000000
50%,642.000000,5.325209e+07,5.332807e+07,5.332065e+07,4.000000
75%,1076.000000,1.050254e+08,1.050706e+08,1.050351e+08,12.000000
max,2484.000000,2.489139e+08,2.489302e+08,2.489139e+08,90.000000
